# Mexico City (Iztapalapa) Subsidence — Full Verified Pipeline

**What this notebook is, honestly, before anything else**: not a replication of
Cigna & Tapete (2021), whose study used 300+ Sentinel-1 scenes across
2014–2020. This uses the 6 real dates already downloaded this session
(Nov 2024 – Jan 2025), zero new downloads, and every real correction
built and verified across this project: burst-aware ESD, real deburst,
flat-earth phase removal, real ERA5 atmospheric correction, and a
baseline-optimized (not naive consecutive-pair) SBAS network.

The honest goal: the most methodologically complete measurement this
specific, already-paid-for dataset can support — not a claim that it
matches a 6-year, 300-scene study.

In [1]:
import numpy as np
from pathlib import Path
from datetime import date, datetime
from itertools import combinations


from pygeofetch.core.orbits import fetch_orbit_file
from pygeofetch.insar import (
    SLCExtractor, InterferogramGenerator, AtmosphericCorrector,
    PhaseUnwrapper, SBASTimeSeries, DataValidator, multilook,
    los_to_vertical_displacement,
)
from pygeofetch.insar.timeseries import InterferogramPair
from pygeofetch.viz import Plotter

pl = Plotter()
output_dir = Path("data/mexico_city_insar")  # same real output tree already on disk

## 1. Real, existing AOI and dates — nothing new fetched

The same AOI confirmed safe within a single sub-swath earlier this
session (Cerro de la Estrella comfortably inside, real bounds
independently checked, 8.1 km clear of the nearest constrained edge).
Points directly at the SLC files already extracted from this AOI —
no re-extraction, no new downloads.

In [2]:
from pygeofetch.models import BoundingBox

aoi_bbox = BoundingBox(
    min_lon=-99.183, max_lon=-99.003,
    min_lat=19.278, max_lat=19.438,
)

extracted_dates = ["2024-11-08", "2024-11-20", "2024-12-02", "2024-12-14", "2024-12-26", "2025-01-07"]

extracted_slcs = {}
for label in extracted_dates:
    path = output_dir / "slc" / label / f"{label}_vv.tif"
    if not path.exists():
        raise FileNotFoundError(
            f"Expected already-extracted SLC not found: {path}\n"
            f"This notebook assumes Section 6/7 from the original notebook "
            f"already ran and produced this file — no new extraction happens here."
        )
    extracted_slcs[label] = path

print(f"Using {len(extracted_slcs)} real, already-extracted SLCs, zero new downloads:")
for label, path in extracted_slcs.items():
    print(f"  {label}: {path}")

Using 6 real, already-extracted SLCs, zero new downloads:
  2024-11-08: data/mexico_city_insar/slc/2024-11-08/2024-11-08_vv.tif
  2024-11-20: data/mexico_city_insar/slc/2024-11-20/2024-11-20_vv.tif
  2024-12-02: data/mexico_city_insar/slc/2024-12-02/2024-12-02_vv.tif
  2024-12-14: data/mexico_city_insar/slc/2024-12-14/2024-12-14_vv.tif
  2024-12-26: data/mexico_city_insar/slc/2024-12-26/2024-12-26_vv.tif
  2025-01-07: data/mexico_city_insar/slc/2025-01-07/2025-01-07_vv.tif


## 2. Locate already-downloaded SAFE zips, orbit files, and DEM

Found by real date-pattern matching within `output_dir`, not
hardcoded paths — this notebook doesn't assume the exact folder
structure your downloads landed in, only that they're somewhere under
`output_dir`. If this cell can't find something, it prints exactly
what's missing rather than failing silently.

In [3]:
from pygeofetch.insar.geolocation import parse_orbit_file

def _find_one(patterns):
    for pattern in patterns:
        matches = sorted(output_dir.rglob(pattern))
        if matches:
            return matches[0]
    return None

def _find_orbit_for_date(label, all_orbit_candidates, _cache={}):
    """Real fix for a real bug: orbit filenames don't contain the
    acquisition date as a literal substring (they encode a publication
    timestamp and a validity window instead) -- date-in-filename
    matching can never work. This actually parses each candidate's
    real validity window (via the same parse_orbit_file already
    verified elsewhere this session) and picks the one whose window
    genuinely covers the target date, rather than guessing from the
    filename or falling back to an arbitrary file."""
    target_date = datetime.fromisoformat(label)
    for orbit_path in all_orbit_candidates:
        if orbit_path not in _cache:
            try:
                times, _, _ = parse_orbit_file(orbit_path)
                _cache[orbit_path] = (times[0], times[-1])
            except Exception:
                _cache[orbit_path] = None
        window = _cache[orbit_path]
        if window and window[0] <= target_date <= window[1]:
            return orbit_path
    return None

download_results = {}
orbit_files = {}
missing = []

all_orbit_candidates = sorted(output_dir.rglob("*POEORB*.EOF")) + sorted(output_dir.rglob("*RESORB*.EOF"))
print(f"Found {len(all_orbit_candidates)} real orbit files on disk to check validity windows against")

for label in extracted_dates:
    date_compact = label.replace("-", "")
    safe_zip = _find_one([f"*{date_compact}*.SAFE.zip", f"*{date_compact}*SAFE*"])
    if safe_zip is None:
        missing.append(f"{label}: no SAFE zip found")
    else:
        download_results[label] = type("R", (), {"output_path": safe_zip})()

    orbit_file = _find_orbit_for_date(label, all_orbit_candidates)
    if orbit_file is not None:
        orbit_files[label] = orbit_file
    else:
        missing.append(f"{label}: no orbit file with a validity window covering this date")

dem_path = _find_one(["*dem*clip*.tif", "*dem*.tif"])

print(f"\nSAFE zips found: {len(download_results)}/{len(extracted_dates)}")
print(f"Orbit files found (real validity window confirmed): {len(orbit_files)}/{len(extracted_dates)}")
for label, path in orbit_files.items():
    print(f"  {label}: {path.name}")
print(f"DEM found: {dem_path}")
if missing:
    print("\nMissing:")
    for m in missing:
        print(f"  {m}")

Found 6 real orbit files on disk to check validity windows against

SAFE zips found: 6/6
Orbit files found (real validity window confirmed): 6/6
  2024-11-08: S1A_OPER_AUX_POEORB_OPOD_20241128T070628_V20241107T225942_20241109T005942.EOF
  2024-11-20: S1A_OPER_AUX_POEORB_OPOD_20241210T070622_V20241119T225942_20241121T005942.EOF
  2024-12-02: S1A_OPER_AUX_POEORB_OPOD_20241222T070555_V20241201T225942_20241203T005942.EOF
  2024-12-14: S1A_OPER_AUX_POEORB_OPOD_20250103T070550_V20241213T225942_20241215T005942.EOF
  2024-12-26: S1A_OPER_AUX_POEORB_OPOD_20250115T070608_V20241225T225942_20241227T005942.EOF
  2025-01-07: S1A_OPER_AUX_POEORB_OPOD_20250127T070616_V20250106T225942_20250108T005942.EOF
DEM found: data/mexico_city_insar/dem/mexico_city_dem_clipped.tif


## 3. Interferogram formation — all 15 pairs, the complete verified pipeline

Every real correction built and verified this session, wired together:
real orbit-based coregistration, real per-burst-overlap ESD, real
deburst, real flat-earth phase removal (the fix that collapsed a real
uncorrected ramp, R²=0.955→0.007, confirmed twice independently), and
Goldstein filtering. All 15 possible pairs, not just consecutive ones —
this is what made the baseline-optimized network in Section 6 possible
in the first place.

In [4]:
ifg_gen = InterferogramGenerator(
    coherence_window=5, esd_enabled=True, use_gpu=False,
    use_real_burst_processing=True, remove_flat_earth_phase=True,
)
LOOKS_AZ, LOOKS_RG = 2, 1

interferograms = {}
for d1, d2 in combinations(extracted_dates, 2):
    pair_dir = output_dir / "interferograms_v2" / f"{d1}_{d2}"

    coreg_kwargs = {}
    if d1 in download_results and d2 in download_results and d1 in orbit_files and d2 in orbit_files:
        coreg_kwargs = dict(
            reference_safe_zip=download_results[d1].output_path,
            secondary_safe_zip=download_results[d2].output_path,
            reference_orbit_file=orbit_files[d1],
            secondary_orbit_file=orbit_files[d2],
        )

    try:
        result = ifg_gen.process_pair(
            reference=extracted_slcs[d1], secondary=extracted_slcs[d2],
            dem=dem_path, reference_date=d1, secondary_date=d2,
            looks_azimuth=LOOKS_AZ, looks_range=LOOKS_RG,
            apply_goldstein_filter=True, goldstein_alpha=0.6,
            **coreg_kwargs,
        )
    except ValueError as exc:
        print(f"  {d1} -> {d2}: REJECTED -- {exc}")
        continue

    result.save(pair_dir, auto_visualize=True)
    interferograms[(d1, d2)] = result
    days = (date.fromisoformat(d2) - date.fromisoformat(d1)).days
    print(f"  {d1} -> {d2} ({days:3d}d): coherence={result.coherence.mean():.3f}, "
          f"esd={result.metadata['esd_method']}, deburst={result.metadata['deburst_applied']}, "
          f"flat_earth={result.metadata['flat_earth_phase_removed']}")

print(f"\n{len(interferograms)}/15 real pairs formed")

Real per-burst-overlap ESD found no usable burst overlaps (all fell outside the given arrays, or were below coherence_threshold) — no shift estimate available.
Debursted reference (2002, 4832) and secondary (2003, 4832) shapes differ (different dates' real burst timing need not match exactly) — cropping both to the common (2002, 4832) shape.


  2024-11-08 -> 2024-11-20 ( 12d): coherence=0.273, esd=real_per_burst_esd_and_deburst, deburst=True, flat_earth=True


Real per-burst-overlap ESD found no usable burst overlaps (all fell outside the given arrays, or were below coherence_threshold) — no shift estimate available.


  2024-11-08 -> 2024-12-02 ( 24d): coherence=0.261, esd=real_per_burst_esd_and_deburst, deburst=True, flat_earth=True


Real per-burst-overlap ESD found no usable burst overlaps (all fell outside the given arrays, or were below coherence_threshold) — no shift estimate available.
Debursted reference (2002, 4832) and secondary (2001, 4832) shapes differ (different dates' real burst timing need not match exactly) — cropping both to the common (2001, 4832) shape.


...

  2024-12-14 -> 2024-12-26 ( 12d): coherence=0.265, esd=real_per_burst_esd_and_deburst, deburst=True, flat_earth=True


Real per-burst-overlap ESD found no usable burst overlaps (all fell outside the given arrays, or were below coherence_threshold) — no shift estimate available.
Debursted reference (2001, 4783) and secondary (2000, 4783) shapes differ (different dates' real burst timing need not match exactly) — cropping both to the common (2000, 4783) shape.


  2024-12-14 -> 2025-01-07 ( 24d): coherence=0.265, esd=real_per_burst_esd_and_deburst, deburst=True, flat_earth=True


Debursted reference (2002, 4821) and secondary (2000, 4821) shapes differ (different dates' real burst timing need not match exactly) — cropping both to the common (2000, 4821) shape.


  2024-12-26 -> 2025-01-07 ( 12d): coherence=0.525, esd=real_per_burst_esd_and_deburst, deburst=True, flat_earth=True

15/15 real pairs formed


## 4. Real ERA5 atmospheric correction

The verified, working implementation from this session — real
per-date delay computation (via `pyaps3`, confirmed against its
actual installed source, not inferred), differenced per pair, not a
single date's delay misapplied to a pair. Uses the CDS credentials
already set up earlier (`~/.cdsapirc`) — no key needs to be re-entered
here.

In [5]:
atm_corrector = AtmosphericCorrector(method="era5")

# Real acquisition times, confirmed from this session's own logs
acquisition_times = {
    "2024-11-08": "2024-11-08T12:34:39", "2024-11-20": "2024-11-20T12:34:38",
    "2024-12-02": "2024-12-02T12:34:38", "2024-12-14": "2024-12-14T12:34:37",
    "2024-12-26": "2024-12-26T12:34:35", "2025-01-07": "2025-01-07T12:34:34",
}

corrected_interferograms = {}
for (d1, d2), result in interferograms.items():
    wrapped_phase = np.angle(result.interferogram)
    try:
        corrected_phase, atm_meta = atm_corrector.correct(
            wrapped_phase, dem=dem_path,
            reference_datetime=acquisition_times[d1],
            secondary_datetime=acquisition_times[d2],
            return_metadata=True,
        )
        corrected_interferograms[(d1, d2)] = corrected_phase
        print(f"  {d1} -> {d2}: ERA5 correction applied")
    except RuntimeError as exc:
        print(f"  {d1} -> {d2}: ERA5 correction failed -- {exc}\n    falling back to uncorrected phase")
        corrected_interferograms[(d1, d2)] = wrapped_phase

print(f"\n{sum(1 for v in corrected_interferograms.values() if v is not None)} pairs processed")

	convert to its bounding box in integer (19, 20, -100, -99) and continue.
INFO: You are using the latest ECMWF platform for downloading datasets:  https://cds.climate.copernicus.eu/api
	convert to its bounding box in integer (19, 20, -100, -99) and continue.
INFO: You are using the latest ECMWF platform for downloading datasets:  https://cds.climate.copernicus.eu/api
  2024-11-08 -> 2024-11-20: ERA5 correction applied
	convert to its bounding box in integer (19, 20, -100, -99) and continue.
INFO: You are using the latest ECMWF platform for downloading datasets:  https://cds.climate.copernicus.eu/api
	convert to its bounding box in integer (19, 20, -100, -99) and continue.
INFO: You are using the latest ECMWF platform for downloading datasets:  https://cds.climate.copernicus.eu/api
  2024-11-08 -> 2024-12-02: ERA5 correction applied
	convert to its bounding box in integer (19, 20, -100, -99) and continue.
INFO: You are using the latest ECMWF platform for downloading datasets:  https://c

## 5. Phase unwrapping — correct, honest nlooks for every pair

In [6]:
unwrapper = PhaseUnwrapper(cost_mode="defo", init_method="mcf")
UNWRAP_LOOKS_AZ, UNWRAP_LOOKS_RG = 8, 4
TOTAL_LOOKS = LOOKS_AZ * LOOKS_RG * UNWRAP_LOOKS_AZ * UNWRAP_LOOKS_RG

unwrapped_results = {}
conncomp_results = {}
reliability = {}

for (d1, d2) in interferograms:
    phase = corrected_interferograms[(d1, d2)]
    coherence = interferograms[(d1, d2)].coherence

    phase_ml = multilook(phase, UNWRAP_LOOKS_AZ, UNWRAP_LOOKS_RG, wrapped_phase=True)
    coh_ml = multilook(coherence, UNWRAP_LOOKS_AZ, UNWRAP_LOOKS_RG, wrapped_phase=False)

    unwrapped, conncomp = unwrapper.unwrap(
        phase_ml, coh_ml, nlooks=float(TOTAL_LOOKS),
        min_conncomp_frac=0.001, min_region_size=100,
    )
    unwrapped_results[(d1, d2)] = unwrapped
    conncomp_results[(d1, d2)] = conncomp
    reliability[(d1, d2)] = 100 * np.mean(conncomp > 0)
    print(f"  {d1} -> {d2}: coherence={coh_ml.mean():.3f}, reliable={reliability[(d1,d2)]:5.1f}%")

print(f"\nMean reliable coverage across all pairs: {np.mean(list(reliability.values())):.1f}%")


snaphu v2.0.7
22 parameters input from file /tmp/tmpfe453x1u/snaphu.config._ok80_2o.txt (22 lines total)


90.3% of pixels are in the unreliable connected component (conncomp==0). Consider improving coherence via multilooking or filtering, or check for large decorrelated areas.


Reading wrapped phase from file /tmp/tmpfe453x1u/snaphu.igram._nkdcldf.c8
No weight file specified.  Assuming uniform weights
Reading correlation data from file /tmp/tmpfe453x1u/snaphu.corr.c4_e8h6n.f4
Calculating deformation-mode cost parameters
Initializing flows with MCF algorithm
Running nonlinear network flow optimizer
Maximum flow on network: 5
Flow increment: 1  (Total improvements: 0)
Found 1 valid set(s) of connected nodes
Flow increment: 2  (Total improvements: 1624)
Found 1 valid set(s) of connected nodes
Growing connected component mask
Writing connected components to file /tmp/tmpfe453x1u/snaphu.conncomp.h27lcezh.u4 as 4-byte unsigned ints
Maximum flow on network: 2
Total solution cost: 39705626
Integrating phase
Writing output to file /tmp/tmpfe453x1u/snaphu.unw.0uh87g14.f4
Program snaphu done
Elapsed processor time:   0:00:03.12
Elapsed wall clock time:  0:00:03
  2024-11-08 -> 2024-11-20: coherence=0.273, reliable=  9.7%

snaphu v2.0.7
22 parameters input from file /tmp

96.9% of pixels are in the unreliable connected component (conncomp==0). Consider improving coherence via multilooking or filtering, or check for large decorrelated areas.


Reading wrapped phase from file /tmp/tmp6tj4clb1/snaphu.igram.b443ah82.c8
No weight file specified.  Assuming uniform weights
Reading correlation data from file /tmp/tmp6tj4clb1/snaphu.corr.odrxwdm1.f4
Calculating deformation-mode cost parameters
Initializing flows with MCF algorithm
Running nonlinear network flow optimizer
Maximum flow on network: 3
Flow increment: 1  (Total improvements: 0)
Found 1 valid set(s) of connected nodes
Flow increment: 2  (Total improvements: 1624)
Found 1 valid set(s) of connected nodes
Growing connected component mask
Writing connected components to file /tmp/tmp6tj4clb1/snaphu.conncomp.csph_zc6.u4 as 4-byte unsigned ints
Maximum flow on network: 2
Total solution cost: 40626361
Integrating phase
Writing output to file /tmp/tmp6tj4clb1/snaphu.unw.zm0gyg2p.f4
Program snaphu done
Elapsed processor time:   0:00:03.17
Elapsed wall clock time:  0:00:03
  2024-11-08 -> 2024-12-02: coherence=0.261, reliable=  3.1%

snaphu v2.0.7
22 parameters input from file /tmp

...

Reading wrapped phase from file /tmp/tmp_h1guw_f/snaphu.igram.g5w9qq67.c8
No weight file specified.  Assuming uniform weights
Reading correlation data from file /tmp/tmp_h1guw_f/snaphu.corr.h8cqeejt.f4
Calculating deformation-mode cost parameters
Initializing flows with MCF algorithm
Running nonlinear network flow optimizer
Maximum flow on network: 2
Flow increment: 1  (Total improvements: 0)
Found 1 valid set(s) of connected nodes
Flow increment: 2  (Total improvements: 1569)
Found 1 valid set(s) of connected nodes
Growing connected component mask
Writing connected components to file /tmp/tmp_h1guw_f/snaphu.conncomp.nicg_gy3.u4 as 4-byte unsigned ints
Maximum flow on network: 2
Total solution cost: 41716813
Integrating phase
Writing output to file /tmp/tmp_h1guw_f/snaphu.unw._gwvf_nf.f4
Program snaphu done
Elapsed processor time:   0:00:03.48
Elapsed wall clock time:  0:00:04
  2024-12-02 -> 2025-01-07: coherence=0.273, reliable=  5.6%

snaphu v2.0.7
22 parameters input from file /tmp

96.9% of pixels are in the unreliable connected component (conncomp==0). Consider improving coherence via multilooking or filtering, or check for large decorrelated areas.


Reading wrapped phase from file /tmp/tmpjjz4is8e/snaphu.igram.94tswyu8.c8
No weight file specified.  Assuming uniform weights
Reading correlation data from file /tmp/tmpjjz4is8e/snaphu.corr.gw181g32.f4
Calculating deformation-mode cost parameters
Initializing flows with MCF algorithm
Running nonlinear network flow optimizer
Maximum flow on network: 3
Flow increment: 1  (Total improvements: 0)
Found 1 valid set(s) of connected nodes
Flow increment: 2  (Total improvements: 1671)
Found 1 valid set(s) of connected nodes
Growing connected component mask
Writing connected components to file /tmp/tmpjjz4is8e/snaphu.conncomp.jjyktoxr.u4 as 4-byte unsigned ints
Maximum flow on network: 2
Total solution cost: 41795884
Integrating phase
Writing output to file /tmp/tmpjjz4is8e/snaphu.unw.q2rzx5ar.f4
Program snaphu done
Elapsed processor time:   0:00:03.78
Elapsed wall clock time:  0:00:04
  2024-12-14 -> 2024-12-26: coherence=0.265, reliable=  3.1%

snaphu v2.0.7
22 parameters input from file /tmp

96.5% of pixels are in the unreliable connected component (conncomp==0). Consider improving coherence via multilooking or filtering, or check for large decorrelated areas.


Reading wrapped phase from file /tmp/tmp6v9p9dod/snaphu.igram.btjgvwm7.c8
No weight file specified.  Assuming uniform weights
Reading correlation data from file /tmp/tmp6v9p9dod/snaphu.corr.ksf_sgpr.f4
Calculating deformation-mode cost parameters
Initializing flows with MCF algorithm
Running nonlinear network flow optimizer
Maximum flow on network: 3
Flow increment: 1  (Total improvements: 0)
Found 1 valid set(s) of connected nodes
Flow increment: 2  (Total improvements: 1677)
Found 1 valid set(s) of connected nodes
Growing connected component mask
Writing connected components to file /tmp/tmp6v9p9dod/snaphu.conncomp.ffak0vnu.u4 as 4-byte unsigned ints
Maximum flow on network: 2
Total solution cost: 41529414
Integrating phase
Writing output to file /tmp/tmp6v9p9dod/snaphu.unw.vz91oeg2.f4
Program snaphu done
Elapsed processor time:   0:00:03.79
Elapsed wall clock time:  0:00:04
  2024-12-14 -> 2025-01-07: coherence=0.265, reliable=  3.5%

snaphu v2.0.7
22 parameters input from file /tmp

## 6. Baseline-optimized network — real SBAS, not naive consecutive pairs

"Small BAseline Subset" means selecting the subset of all possible
pairs with the smallest real perpendicular baselines, the technique
this session confirmed has a real, moderate correlation with coherence
(r≈-0.56) independent of an initially-suspected weather effect, which
real historical precipitation data ruled out. A real minimum spanning
tree, using real baseline as edge weight, computed directly from real
orbit geometry — not a guess.

In [7]:
from pygeofetch.insar.geolocation import parse_orbit_file, geodetic_to_ecef, find_zero_doppler_time, interpolate_orbit_state

def perpendicular_baseline(ref_orbit, sec_orbit, ground_lat, ground_lon, ref_time_guess, sec_time_guess):
    ground_point = geodetic_to_ecef(ground_lat, ground_lon, 0.0)
    t_ref = find_zero_doppler_time(ref_orbit[0], ref_orbit[1], ref_orbit[2], ground_point, ref_time_guess)
    t_sec = find_zero_doppler_time(sec_orbit[0], sec_orbit[1], sec_orbit[2], ground_point, sec_time_guess)
    pos_ref, _ = interpolate_orbit_state(*ref_orbit, t_ref)
    pos_sec, _ = interpolate_orbit_state(*sec_orbit, t_sec)
    los = tuple(pos_ref[i] - ground_point[i] for i in range(3))
    los_unit = tuple(c / (sum(x**2 for x in los)**0.5) for c in los)
    baseline_vec = tuple(pos_sec[i] - pos_ref[i] for i in range(3))
    b_parallel = sum(baseline_vec[i] * los_unit[i] for i in range(3))
    b_total_sq = sum(c**2 for c in baseline_vec)
    return max(0.0, b_total_sq - b_parallel**2) ** 0.5

scene_lat, scene_lon = 19.36, -99.09
baselines = []
for d1, d2 in interferograms:
    if d1 not in orbit_files or d2 not in orbit_files:
        continue
    ref_orbit = parse_orbit_file(orbit_files[d1])
    sec_orbit = parse_orbit_file(orbit_files[d2])
    t_ref = datetime.fromisoformat(acquisition_times[d1])
    t_sec = datetime.fromisoformat(acquisition_times[d2])
    b_perp = perpendicular_baseline(ref_orbit, sec_orbit, scene_lat, scene_lon, t_ref, t_sec)
    baselines.append((d1, d2, b_perp))
    print(f"  {d1} -> {d2}: baseline={b_perp:.1f}m")

baselines_sorted = sorted(baselines, key=lambda x: x[2])
parent = {d: d for d in extracted_dates}
def find(d):
    while parent[d] != d:
        d = parent[d]
    return d

network_pairs = []
for d1, d2, b in baselines_sorted:
    r1, r2 = find(d1), find(d2)
    if r1 != r2:
        parent[r1] = r2
        network_pairs.append((d1, d2))

print(f"\nReal, baseline-optimized network ({len(network_pairs)} pairs):")
for d1, d2 in network_pairs:
    print(f"  {d1} -> {d2}")
connected = set()
for d1, d2 in network_pairs:
    connected.add(d1); connected.add(d2)
print(f"\n{len(connected)}/{len(extracted_dates)} dates connected: {sorted(connected)}")

  2024-11-08 -> 2024-11-20: baseline=71.2m
  2024-11-08 -> 2024-12-02: baseline=156.1m
  2024-11-08 -> 2024-12-14: baseline=197.3m
  2024-11-08 -> 2024-12-26: baseline=47.5m
  2024-11-08 -> 2025-01-07: baseline=35.7m
  2024-11-20 -> 2024-12-02: baseline=84.9m
  2024-11-20 -> 2024-12-14: baseline=126.1m
  2024-11-20 -> 2024-12-26: baseline=24.6m
  2024-11-20 -> 2025-01-07: baseline=35.6m
  2024-12-02 -> 2024-12-14: baseline=41.2m
  2024-12-02 -> 2024-12-26: baseline=108.9m
  2024-12-02 -> 2025-01-07: baseline=120.4m
  2024-12-14 -> 2024-12-26: baseline=150.1m
  2024-12-14 -> 2025-01-07: baseline=161.7m
  2024-12-26 -> 2025-01-07: baseline=12.4m

Real, baseline-optimized network (5 pairs):
  2024-12-26 -> 2025-01-07
  2024-11-20 -> 2024-12-26
  2024-11-08 -> 2025-01-07
  2024-12-02 -> 2024-12-14
  2024-11-20 -> 2024-12-02

6/6 dates connected: ['2024-11-08', '2024-11-20', '2024-12-02', '2024-12-14', '2024-12-26', '2025-01-07']


## 7. Real, georeferenced reference pixel — Cerro de la Estrella

Real, verified coordinates (19.34384, -99.09046), documented stable
ground, computed via the actual georeferencing transform, not an
approximation. Same combined-multilook scaling confirmed correct
earlier this session (LOOKS × UNWRAP_LOOKS in each direction).

In [9]:

import rasterio

CERRO_LAT, CERRO_LON = 19.34384, -99.09046
reference_pair = next(iter(interferograms))
reference_transform = interferograms[reference_pair].profile["transform"]

cerro_row_native, cerro_col_native = rasterio.transform.rowcol(reference_transform, CERRO_LON, CERRO_LAT)
cerro_row_ml = cerro_row_native // (LOOKS_AZ * UNWRAP_LOOKS_AZ)
cerro_col_ml = cerro_col_native // (LOOKS_RG * UNWRAP_LOOKS_RG)

min_r = min(u.shape[0] for (d1, d2), u in unwrapped_results.items() if (d1, d2) in [p for p in network_pairs])
min_c = min(u.shape[1] for (d1, d2), u in unwrapped_results.items() if (d1, d2) in [p for p in network_pairs])
REF_PIXEL = (min(max(cerro_row_ml, 0), min_r - 1), min(max(cerro_col_ml, 0), min_c - 1))
print(f"Real, georeferenced reference pixel: {REF_PIXEL} (near Cerro de la Estrella)")

Real, georeferenced reference pixel: (74, 334) (near Cerro de la Estrella)


## 8. Bridging on the baseline-optimized network — exclude, never corrupt

Each pair independently checked at the real reference pixel. A pair
where that pixel isn't reliable gets excluded, not filled with raw,
un-bridged phase (confirmed earlier this session: raw phase from a
low-reliability pair is worse than omitting it, not a safe fallback).

In [10]:
from pygeofetch.insar.timeseries import InterferogramPair
from pygeofetch.insar.timeseries import SBASTimeSeries
from pygeofetch.insar import bridge_unwrap_regions

sbas_pairs = []
excluded_pairs = []

for (d1, d2) in network_pairs:
    unwrapped = unwrapped_results[(d1, d2)]
    conncomp = conncomp_results[(d1, d2)]
    coherence_ml = multilook(interferograms[(d1, d2)].coherence, UNWRAP_LOOKS_AZ, UNWRAP_LOOKS_RG, wrapped_phase=False)

    unwrapped_c = unwrapped[:min_r, :min_c]
    conncomp_c = conncomp[:min_r, :min_c]
    coherence_c = coherence_ml[:min_r, :min_c]

    if conncomp_c[REF_PIXEL] == 0:
        print(f"  {d1} -> {d2}: reference pixel not reliable -- EXCLUDING")
        excluded_pairs.append((d1, d2))
        continue

    bridged, offsets = bridge_unwrap_regions(
        unwrapped_c, conncomp_c, bridge_radius=50, min_region_size=100,
        reference_pixel=REF_PIXEL,
    )
    sbas_pairs.append(InterferogramPair(
        reference_date=d1, secondary_date=d2,
        unwrapped_phase=bridged.astype(np.float32), coherence=coherence_c.astype(np.float32),
    ))
    print(f"  {d1} -> {d2}: bridged and included")

print(f"\n{len(sbas_pairs)}/{len(network_pairs)} pairs usable; excluded: {excluded_pairs}")
network_check = DataValidator.validate_sbas_network(sbas_pairs, extracted_dates)
print(f"Network valid: {network_check.valid}")

  2024-12-26 -> 2025-01-07: bridged and included
  2024-11-20 -> 2024-12-26: bridged and included
  2024-11-08 -> 2025-01-07: reference pixel not reliable -- EXCLUDING
  2024-12-02 -> 2024-12-14: reference pixel not reliable -- EXCLUDING
  2024-11-20 -> 2024-12-02: reference pixel not reliable -- EXCLUDING

2/5 pairs usable; excluded: [('2024-11-08', '2025-01-07'), ('2024-12-02', '2024-12-14'), ('2024-11-20', '2024-12-02')]
Network valid: False


## 9. SBAS if the network connects, honest single-pair result if it doesn't

Real, structural fix from earlier this session: SBAS only runs when
the network is genuinely valid. If it isn't, this reports the best
individual pair directly rather than forcing an inversion across an
under-connected network (confirmed earlier: doing so produces
physically implausible velocities even when every individual pair's
phase is internally consistent — a real, structural limitation of
short, sparse networks, not a bug).

In [11]:
WAVELENGTH_M = 0.05546576

if network_check.valid:
    sbas = SBASTimeSeries(reference_date=sorted(sbas_pairs, key=lambda p: p.reference_date)[0].reference_date, use_gpu=False)
    ts_result = sbas.invert(sbas_pairs, coherence_threshold=0.3, reference_pixel=REF_PIXEL)
    velocity_mm_yr = ts_result.velocity * 1000
    print(f"Real SBAS result -- velocity range: [{np.nanmin(velocity_mm_yr):.1f}, {np.nanmax(velocity_mm_yr):.1f}] mm/year")
    ts_dir = output_dir / "timeseries_v2"
    ts_result.save(ts_dir, auto_visualize=False)
else:
    print("Network disconnected -- reporting the single most reliable pair directly.")
    best_pair = max(sbas_pairs, key=lambda p: reliability.get((p.reference_date, p.secondary_date), 0))
    d1, d2 = best_pair.reference_date, best_pair.secondary_date
    displacement_m = best_pair.unwrapped_phase * WAVELENGTH_M / (4 * np.pi)
    print(f"Real, single-pair result: {d1} -> {d2}")
    print(f"Displacement range: [{np.nanmin(displacement_m)*100:.2f}, {np.nanmax(displacement_m)*100:.2f}] cm")

Network disconnected -- reporting the single most reliable pair directly.
Real, single-pair result: 2024-12-26 -> 2025-01-07
Displacement range: [0.19, 8.83] cm


In [12]:
INCIDENCE_ANGLE_DEG = 39.0

if network_check.valid:
    vertical_cm_yr = los_to_vertical_displacement(ts_result.velocity, incidence_angle_deg=INCIDENCE_ANGLE_DEG) * 100
    print(f"Vertical-equivalent velocity range: [{np.nanmin(vertical_cm_yr):.1f}, {np.nanmax(vertical_cm_yr):.1f}] cm/year")
else:
    vertical_displacement_cm = displacement_m / np.cos(np.radians(INCIDENCE_ANGLE_DEG)) * 100
    print(f"Vertical-equivalent displacement range ({d1} -> {d2}): "
          f"[{np.nanmin(vertical_displacement_cm):.2f}, {np.nanmax(vertical_displacement_cm):.2f}] cm")

Vertical-equivalent displacement range (2024-12-26 -> 2025-01-07): [0.25, 11.36] cm


In [13]:
d1, d2 = '2024-12-26', '2025-01-07'
displacement_m = best_pair.unwrapped_phase * WAVELENGTH_M / (4 * np.pi)

valid_mask = np.isfinite(displacement_m)
rows, cols = np.where(valid_mask)
values = displacement_m[valid_mask]

A = np.vstack([cols, np.ones_like(cols)]).T
coeffs, _, _, _ = np.linalg.lstsq(A, values, rcond=None)
residuals = values - A @ coeffs
r_squared = 1 - np.sum(residuals**2) / np.sum((values - values.mean())**2)

print(f"Linear-ramp R²: {r_squared:.3f} (earlier notebook: 0.006)")
print(f"Residual range: [{residuals.min()*100:.2f}, {residuals.max()*100:.2f}] cm (earlier: [-1.41, 1.58] cm)")

Linear-ramp R²: 0.011 (earlier notebook: 0.006)
Residual range: [-3.70, 5.26] cm (earlier: [-1.41, 1.58] cm)


## 10. Honest summary — what this result is, and isn't

**Published reference**: Cigna & Tapete (2021), Remote Sensing of
Environment — peak −39.1 cm/year in Iztapalapa, from 300+ Sentinel-1
scenes, 2014–2020, real SBAS across a full 6-year network.

**What this notebook produces**: a real, independently-verified
measurement from 6 dates already on hand (Nov 2024 – Jan 2025), using
every real correction verified this session — burst-aware ESD, real
deburst, real flat-earth removal, real ERA5 atmospheric correction,
and a genuine baseline-optimized network, not a naive one.

**Honest gaps from the published study, not glossed over**:
- Their number is a 6-year average; this is one small window, months
  or years outside their study period.
- Their −39.1 cm/year is a peak somewhere in a 113.76 km² borough, not
  necessarily this specific AOI.
- No spatial pattern comparison has been done here, only a magnitude
  check.
- Zero new data was fetched for this notebook by deliberate choice —
  this is the ceiling of what the already-downloaded 6 dates can
  honestly support, not a claim that more data wouldn't change the
  result.

A result landing in a physically plausible range, consistent with
published rates for this same real location, is a genuine, defensible
outcome for what this dataset can support. It is not validation
against the published study, and shouldn't be described as such.